# Experiment 6: From-Scratch NumPy GloVe & Statistical N-gram Language Model

Weighted least-squares GloVe training and Laplace-smoothed bigram/trigram prediction.

**Author:** Somansh Goel (SAP ID: 500123667, Roll No.: R2142230744, AIML Batch 1)


In [ ]:
import numpy as np
from collections import Counter, defaultdict

text = (
    "the king and the queen rule the royal kingdom "
    "the prince and the princess live in the royal palace "
    "natural language processing enables computers to understand natural language "
    "natural language processing powers modern artificial intelligence systems "
    "the king leads the kingdom with wisdom and the queen guides the royal court"
)
words = text.split()
vocab = sorted(set(words))
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
cooc = defaultdict(float)
for idx, w in enumerate(words):
    for j in range(max(0, idx-2), min(len(words), idx+3)):
        if idx != j:
            cooc[(w2i[w], w2i[words[j]])] += 1.0 / abs(idx - j)

rng = np.random.default_rng(42)
W, Wc = rng.normal(0, 0.1, (V, 50)), rng.normal(0, 0.1, (V, 50))
b, bc = np.zeros(V), np.zeros(V)
for ep in range(15):
    loss = 0.0
    for (i, j), x in cooc.items():
        wt = (x / 10.0) ** 0.75 if x < 10.0 else 1.0
        diff = float(np.dot(W[i], Wc[j]) + b[i] + bc[j] - np.log(x))
        loss += wt * diff * diff
        g = 2.0 * wt * diff
        W[i] -= 0.05 * g * Wc[j]; Wc[j] -= 0.05 * g * W[i]
        b[i] -= 0.05 * g; bc[j] -= 0.05 * g
    if ep in (0, 14):
        print(f"Epoch {ep+1} loss: {loss:.4f}")
